In [ ]:
# Imports
import pandas as pd
import numpy as np

import os
from datetime import datetime

# Base directory for the original data (lambda function for dynamic date range)
base_dir   = lambda date: f"./data/export_{date}"

dates_list = ['3.12.16-4.11.16', '4.12.16-5.12.16']

original_activity1 = pd.read_csv(f'{base_dir(dates_list[0])}/dailyActivity.csv')
original_activity2 = pd.read_csv(f'{base_dir(dates_list[1])}/dailyActivity.csv')

# Lo datos trabajan con mets de 6 como intensidad 3 como último nivel, siende que podría generar más grados de este para deportistas o acciones simliares. Identificando estos parametros puedo concluir que el registro de parametros no está enfocada a deportistas. Pero sí a usuarios que se encuentra en reposo hasta actividades moderados, como trote o baile


In [ ]:
# Limit data range from dataframe
def limit_date_range(df, date_range):
    sep_dates = date_range.split('-')
    initial_date = str(datetime.strptime(sep_dates[0], '%m.%d.%y').date())
    last_date    = str(datetime.strptime(sep_dates[1], '%m.%d.%y').date())

    range = df[(df['ActivityDay'] >= initial_date) & (df['ActivityDay'] <= last_date)]

    return range

# Merge dataframes
def connect_dailyActivity_files():
    df_dailyActivity = pd.read_csv(f'{base_dir(dates_list[0])}/dailyActivity.csv')
    df_dailyActivity = limit_date_range(df_dailyActivity, dates_list[0])
    for date in dates_list[1:]:
        df_con = pd.read_csv(f'{base_dir(date)}/dailyActivity.csv')
        df_con = limit_date_range(df_con, date)
        df_dailyActivity = pd.merge(
                df_dailyActivity, 
                df_con, 
                how='outer'
            )
    return df_dailyActivity

merged_dailyActivity = connect_dailyActivity_files()

merged_dailyActivity['ActivityDay'] = pd.to_datetime(merged_dailyActivity['ActivityDay'])


#  Columna con el número de la semana según el estándar ISO 8601
merged_dailyActivity['Week_ISO'] = merged_dailyActivity['ActivityDay'].dt.isocalendar().week

# https://www.jacc.org/doi/10.1016/j.jacc.2023.07.029
# Actividad saludables según los pasos


# https://iris.who.int/items/65310979-92e8-4c98-8092-5a16ca07fc2f
# Actividad recomendada entre 150 a 300 minutos de actividad moderada semanales para persona de 18 a 64 años
# de momento se usara el punto medio que sería 225 para el punto optimo y también el minimo de 150. estos serán separados en a lo largo de los 7 días de la semana

# *** 1. Definición de Metas Diarias (Basado en OMS) ***
# Minuto de salud mínimo (150 min/semana / 7 días ≈ 21.4 minutos)
MIN_HEALTH_GOAL_DAILY = 22 

# Meta Óptima (Tu punto medio de 225 min/semana / 7 días ≈ 32.14 minutos)
OPTIMAL_GOAL_DAILY = 32

# Meta Óptima (Tu punto medio de 225 min/semana / 7 días ≈ 32.14 minutos)
BEST_GOAL_DAILY = 43

# *** 2. Cálculo del Puntaje Combinado (Activity Score) ***
# The Pondered Score reflects the effective volume of exercise.
# 1 minute Vigorosa (Int 3) = 2 minutes Moderada (Int 2)
merged_dailyActivity['Combined_Activity_Score'] = (
    merged_dailyActivity['VeryActiveMinutes'] * 2
) + merged_dailyActivity['ModeratelyActiveMinutes']


# --- 3. Creación de Flags de Cumplimiento ---

# Flag 0: Compliance with less than  Minimum Health Goal (22 minutes equivalent)
# This identifies individuals who do not meet the basic requirements for cardiovascular benefits.
merged_dailyActivity['Flag_Sedentary'] = np.where(
    merged_dailyActivity['Combined_Activity_Score'] < MIN_HEALTH_GOAL_DAILY, 
    1, # 1 if True
    0  # 0 if False
)

# Flag 1: Compliance with the Minimum Health Goal (22 minutes equivalent)
# This identifies individuals meeting the basic requirement for cardiovascular benefit.
merged_dailyActivity['Flag_Meets_Min_Health'] = np.where(
    ((merged_dailyActivity['Combined_Activity_Score'] >= MIN_HEALTH_GOAL_DAILY) & (merged_dailyActivity['Combined_Activity_Score'] < OPTIMAL_GOAL_DAILY)), 
    1, # 1 if True
    0  # 0 if False
)

# Flag 2: Compliance with the Optimal Goal (32 minutes equivalent)
# This identifies individuals leading an optimally active lifestyle (your key segment for Bellabeat).
merged_dailyActivity['Flag_Meets_Optimal_Goal'] = np.where(
    ((merged_dailyActivity['Combined_Activity_Score'] >= OPTIMAL_GOAL_DAILY) & (merged_dailyActivity['Combined_Activity_Score'] < BEST_GOAL_DAILY)), 
    # merged_dailyActivity['Combined_Activity_Score'] >= OPTIMAL_GOAL_DAILY, 
    1, # 1 if True
    0  # 0 if False
)

# Flag 3: Compliance with the Best Goal (43 minutes equivalent)
# This identifies individuals leading an active lifestyle (your key segment for Bellabeat).
merged_dailyActivity['Flag_Meets_Best_Goal'] = np.where(
    merged_dailyActivity['Combined_Activity_Score'] >= BEST_GOAL_DAILY, 
    1, # 1 if True
    0  # 0 if False
)

##############################################################################################

merged_dailyActivity['ActivityDay'] = pd.to_datetime(merged_dailyActivity['ActivityDay'])

first_day       = merged_dailyActivity['ActivityDay'].min()
last_day        = merged_dailyActivity['ActivityDay'].max()
diff_days_range = (last_day - first_day).days + 1 

print("*** Datos rango de datos de las fechas ***")

print()

print(first_day)
print(last_day)

print()

print(f'Cantidad de días    : {diff_days_range}')
print(f'Cantidad de horas   : {diff_days_range * 24}')
print(f'Cantidad de minutos : {diff_days_range * 1440}')

print()

merged_dailyActivity

## - Agrupar los registros por el id de los usuarios, para ver los rangos, cantidad de datos

In [ ]:

user_info = merged_dailyActivity.groupby(['Id',]).agg(
    initial_date = ('ActivityDay', 'min'),         # Find the earliest available date
    lastest_date = ('ActivityDay', 'max'), 

    # Días con datos registrados
    count_Records = ('ActivityDay', 'count'), 
    no_Records    = ('ActivityDay', lambda x : diff_days_range - x.count()),

    # Contar días dependiendo de la cantidad de pasos registrados

    median_steps    = ('Steps', 'median'),

    mov_VeryActive  = ('Steps', lambda x: ((x >= 10000)).sum()),
    mov_Moderately  = ('Steps', lambda x: ((x >= 7500) & (x < 10000)).sum()),
    mov_Light       = ('Steps', lambda x: ((x >= 5000) & (x < 7500 )).sum()),
    mov_Sedentary   = ('Steps', lambda x: ((x >= 0   ) & (x < 5000 )).sum()),

    # Días que tienen todos sus datos registrados (1440 minutos)
    full_days       = ('minutes_instances', lambda x: (x == 1440).sum()),
    incomplete_days = ('minutes_instances', lambda x: (x != 1440).sum()),

    # Días que tienen registrado su actividad de sueño y si estos son de  sueño saludable ( mínimo 7 horas)
    days_of_sleep  = ('TotalMinutesAsleep', lambda x: (x > 0).sum()),
    healthy_sleep  = ('TotalMinutesAsleep', lambda x: (x > 420).sum()),

    # Puntos de actividad
    Combined_Activity_Score    = ('Combined_Activity_Score', 'sum'),

    # Días con actividad física y suficiente
    Flag_Meets_Min_Health    = ('Flag_Meets_Min_Health', 'sum'),
    Flag_Meets_Optimal_Goal  = ('Flag_Meets_Optimal_Goal', 'sum'),
    Flag_Meets_Best_Goal     = ('Flag_Meets_Best_Goal', 'sum'),
    Meets_Less_Min_Goal      = ('Flag_Sedentary', 'sum'),
    

    intensities_median    = ('Combined_Activity_Score', 'median'),

)


# user_info['steps_Health'] = 

user_info.head(2)

##  - Definir los usuarios a analizar
Se busca analizar a los usuarios que dispongan la mayor cantidad de datos, se utilizará el rango de 56 días en adelante cubriendo cerca del %90 de uso de los dispositivos que corresponen a una cantidad de 26 usuarios, siendo el cerca del %74 de total. Se dará prioridad a la calidad de datos. 

In [ ]:
## Personas que tienen más de 42 días de registrados
## cantidad de personas entre 0-28, 29-41, 42 - 62  dias de actividad, 


# Categorización de usuarios dependiendo de la cantidad de registros separados por cantidad de semanas
bins = [0,29,42,49,56, 62]

user_activity = user_info.copy()
user_activity['activity_days'] = pd.cut(user_activity['count_Records'], bins = bins)
user_activity.groupby('activity_days', observed=True)['count_Records'].count()

# print(len(user_activity))

### Filtrar los usuarios en el rango definido

In [ ]:
user_activity = user_activity[user_activity['count_Records'] > 56]
user_activity.index.to_list()

merged_dailyActivity = merged_dailyActivity[merged_dailyActivity['Id'].isin(user_activity.index.to_list())]
user_activity.head(1)

---
## - Categorizar usuarios

### Categoría usuarios con dependeindo de la cantidad de pasos frente a la intensidad
Segmentos Clave Identificados (Conteo de Usuarios)

La segmentación cruzada del comportamiento de Pasos e Intensidad dio como resultado los siguientes grupos de usuarios:

| Categoría de Pasos Típicos | Intensidad Típica (Moda) | Conteo de Usuarios | Implicación del Comportamiento |
| :--- | :--- | :--- | :--- |
| **Sedentario/Mínimo** | **Riesgo Crónico** ($< 22 \text{ min}$) | **6 Usuarios** | Inactividad sistémica (fallan el mínimo de salud). |
| **Óptimo/Muy Activo** | **Máximo Rendimiento** ($\ge 43 \text{ min}$) | **15 Usuarios** | Alto compromiso y rendimiento sostenido. |


In [ ]:
###### Usuarios catalogados por pasos frente a intensidad

limites_pasos   = [0, 2735, 4844, 7170, 10000, np.inf] 
etiquetas_pasos = ['Sedentario', 'Mínimo', 'leve', 'Óptimo', 'Muy Activo']

user_activity['healthy_steps'] = pd.cut(
    user_activity['median_steps'], # Columna a clasificar
    bins=limites_pasos,            # Los límites definidos
    labels=etiquetas_pasos,        # Las etiquetas definidas
    right=False                    # El límite izquierdo es inclusivo ([0, 5000))
)

limite_intensidad = [0, MIN_HEALTH_GOAL_DAILY, OPTIMAL_GOAL_DAILY, BEST_GOAL_DAILY, np.inf]

intensities_columns =  ['Baja Intencidad', 'Intencidad Mínima' , 'Intensidad Optima' , 'Intencidad Alta']

user_activity['set_intensities'] = pd.cut(
    user_activity['intensities_median'], # Columna a clasificar
    bins=limite_intensidad,              # Los límites definidos
    labels=intensities_columns,          # Las etiquetas definidas
    right=False                          # El límite izquierdo es inclusivo ([0, 5000))
)

df_segmentation_count = pd.crosstab(
    # La columna que quieres en las filas
    user_activity['healthy_steps'], 
    # La columna que quieres en las columnas
    user_activity['set_intensities'],
    # observed=True si las columnas son categóricas

)
df_segmentation_count


###  El Cliente Eficiente (Discrepancia Clave)

* Mediana de días en Riesgo (36.0): El cliente cuyo comportamiento típico es sedentario pasa la mediana de $\mathbf{36}$ días (de un total de $\approx 62$ días) sin alcanzar el umbral mínimo de salud.

* Mediana de días en Alto Rendimiento (61.0): El cliente cuyo comportamiento típico es dar muchos pasos tiene la mediana de $\mathbf{61}$ días donde logra el máximo esfuerzo.


Se detectó un segmento minoritario donde la **intensidad es alta a pesar de un volumen de pasos bajo**. Este es el **Cliente Eficiente**.

* **Significado:** La intensidad alta sin un alto volumen de Pasos sugiere actividad de **bajo impacto** (ciclismo, pesas, etc.). Esta discrepancia es crucial la campaña de marketing, ya que la métrica de Pasos subestima su esfuerzo real.






In [ ]:

# user_activity['Most_Frequent_Intensity_Goal'] = user_activity[intensities_columns].idxmax(axis=1)

intensities_columns =  ['Meets_Less_Min_Goal', 'Flag_Meets_Min_Health' , 'Flag_Meets_Optimal_Goal' , 'Flag_Meets_Best_Goal'   ]
df_steps_intensities = user_activity.groupby('healthy_steps', observed=True).agg(
    {col: 'median' for col in intensities_columns}
)
df_steps_intensities

### 

### - Cuantificación del Impacto: Volumen Total de Días por Segmento

El siguiente análisis utiliza la **Suma del Total de Días** para cada categoría de cumplimiento de intensidad, lo que permite cuantificar dónde se concentra la **inactividad (Riesgo)** y el **máximo esfuerzo (Rendimiento)** en nuestra muestra.

La tabla cruza la **Clasificación Típica de Pasos** (filas) con la **Suma Total de Días** que los clientes de ese segmento tuvieron en cada nivel de Intensidad.

| Pasos Típicos | Días de Riesgo (< 22 min) | Días de Mínimo (22-31 min) | Días de Óptimo (32-42 min) | Días de Élite (> 43 min) |
| :--- | :--- | :--- | :--- | :--- |
| **Sedentario** | **143** | 25 | 10 | 67 |
| **Mínimo** | **151** | 40 | 27 | 88 |
| **Leve** | 31 | 6 | 13 | 74 |
| **Óptimo** | 38 | 17 | 26 | **404** |
| **Muy Activo** | 22 | 1 | 1 | **402** |

---

### -.1 Cuantificación del Riesgo Crónico (Inactividad)

El mayor volumen de inactividad se concentra en los segmentos de pasos más bajos:

* **Alto Volumen de Inactividad:** Los clientes Sedentarios y Mínimos (Pasos Típicos Bajos) generaron un total de **$\mathbf{294 \text{ días}}$** ($143 + 151$) sin alcanzar siquiera el umbral mínimo de salud.
    > *Este hallazgo cuantifica el problema principal para Bellabeat: la inactividad crónica. Se confirma que el **riesgo de salud** se debe principalmente al **bajo volumen de pasos***.

### -.2. Cuantificación del Rendimiento (Volumen de Élite)

La mayor parte del tiempo de ejercicio de alta calidad proviene del segmento activo:

* **Concentración de la Élite:** Los clientes Óptimos y Muy Activos (Pasos Típicos Altos) generaron un total de **$\mathbf{806 \text{ días}}$** ($404 + 402$) en la categoría de Máximo Rendimiento (`Días de Élite`).
* **Implicación:** Esto demuestra que el volumen de la **actividad de alta intensidad** es producido, casi en su totalidad, por el grupo que ya tiene un alto volumen de pasos, validando la correlación observada.

### -.3. El Cliente Eficiente (Volumen Mínimo, Impacto Fuerte)

Aunque la inactividad de élite se concentra en los segmentos activos, es crucial notar el bajo volumen de días de riesgo en esos segmentos:

* Los clientes **Muy Activos** produjeron solo **$\mathbf{22 \text{ días}}$** de riesgo de inactividad, en comparación con los $\mathbf{143 \text{ días}}$ de los Sedentarios. Esto demuestra que su volumen de inactividad es marginal, reforzando la fiabilidad de este segmento.

In [ ]:
intensities_columns =  ['Meets_Less_Min_Goal', 'Flag_Meets_Min_Health' , 'Flag_Meets_Optimal_Goal' , 'Flag_Meets_Best_Goal'   ]
df_steps_intensities2 = user_activity.groupby('healthy_steps', observed=True).agg(
    {col: 'sum' for col in intensities_columns}
)
df_steps_intensities2

In [ ]:
############################################################################################################################################################

In [ ]:
# Mostrar info de datos
#   Rango de fechas
#   Cantidad de días
#   Total usuarios

# Definir tipos de usurios por:
#   Cantidad de días > x (pro definir) 0
#   Actividad semanal Deportiva
#   Actividad semanal sedentaria (Flag_Meets_Less_Min_Goal)
#

In [ ]:
# calcular si hizo el ejercicio semanal
# definir persona activo - como - cuanto
# definir persona sedentaria - como - cuanto

# merged_dailyActivity.groupby(merged_dailyActivity['ActivityDay'].dt.isocalendar().week)
# merged_dailyActivity['ActivityDay'].dt.isocalendar().week
week_Activity = merged_dailyActivity.groupby(['Id','Week_ISO']).agg(
    initial_date = ('ActivityDay', 'min'),         # Find the earliest available date
    lastest_date = ('ActivityDay', 'max'),

    # # Días con datos registrados
    count_Records = ('ActivityDay', 'count'), 

    # no_Records    = ('ActivityDay', lambda x: ((x.max() - x.min()).days + 1) - x.count()),
    # # no_Records    = ('ActivityDay', lambda x : diff_days_range - x.count()),

    # Contar días dependiendo de la cantidad de pasos registrados
    median_steps    = ('Steps', 'median'),

    mov_VeryActive  = ('Steps', lambda x: ((x >= 10000)).sum()),
    mov_Moderately  = ('Steps', lambda x: ((x >= 7500) & (x < 10000)).sum()),
    mov_Light       = ('Steps', lambda x: ((x >= 5000) & (x < 7500 )).sum()),
    mov_Sedentary   = ('Steps', lambda x: ((x >= 0   ) & (x < 5000 )).sum()),




    # Días que tienen registrado su actividad de sueño y si estos son de  sueño saludable ( mínimo 7 horas)
    days_of_sleep  = ('TotalMinutesAsleep', lambda x: (x > 0).sum()),
    healthy_sleep  = ('TotalMinutesAsleep', lambda x: (x > 420).sum()),
 

    # Puntos de actividad
    Combined_Activity_Score    = ('Combined_Activity_Score', 'sum'),
    Min_Healty_Week_Goal       = ('Combined_Activity_Score', lambda x : x.sum() >= (x.count() * MIN_HEALTH_GOAL_DAILY)),
    Optimal_Goal_week_Goal     = ('Combined_Activity_Score', lambda x : x.sum() >= (x.count() * OPTIMAL_GOAL_DAILY)),
    Best_Goal_week_Goal        = ('Combined_Activity_Score', lambda x : x.sum() >= (x.count() * BEST_GOAL_DAILY)),

    healthy_days = ('Flag_Meets_Best_Goal', lambda x : x.sum() >= 3), # Personas hacen actividad física 
                                                        
    # Días con actividad física y suficiente
    Flag_Meets_Min_Health    = ('Flag_Meets_Min_Health', 'sum'),
    Flag_Meets_Optimal_Goal  = ('Flag_Meets_Optimal_Goal', 'sum'),
    Flag_Meets_Best_Goal     = ('Flag_Meets_Best_Goal', 'sum'),
    Flag_Meets_Less_Min_Goal = ('Flag_Meets_Min_Health', lambda x : (x == 0).sum()),
     
)

# Combined_Activity_Score / dias(coun/sum)
week_Activity

# # *** 1. Definición de Metas Diarias (Basado en OMS) ***
# # Minuto de salud mínimo (150 min/semana / 7 días ≈ 21.4 minutos)
# MIN_HEALTH_GOAL_DAILY = 22 

# # Meta Óptima (Tu punto medio de 225 min/semana / 7 días ≈ 32.14 minutos)
# OPTIMAL_GOAL_DAILY = 32

In [ ]:

print(len(week_Activity[ week_Activity['median_steps'] < 2735]))
print(len(week_Activity[(week_Activity['median_steps'] >=  2517) & (week_Activity['median_steps'] < 4844)]))
print(len(week_Activity[(week_Activity['median_steps'] >=  4844) & (week_Activity['median_steps'] < 7170)]))
print(len(week_Activity[(week_Activity['median_steps'] >=  7170) & (week_Activity['median_steps'] < 10000)]))
print(len(week_Activity[ week_Activity['median_steps'] > 10000]))

In [ ]:
test2 = merged_dailyActivity.groupby(['Week_ISO']).agg(
    initial_date = ('ActivityDay', 'min'),         # Find the earliest available date
    lastest_date = ('ActivityDay', 'max'),

    # # Días con datos registrados
    count_Records = ('ActivityDay', 'count'), 

    # no_Records    = ('ActivityDay', lambda x: ((x.max() - x.min()).days + 1) - x.count()),
    # # no_Records    = ('ActivityDay', lambda x : diff_days_range - x.count()),

    # Contar días dependiendo de la cantidad de pasos registrados
    mov_VeryActive  = ('Steps', lambda x: ((x >= 10000)).sum()),
    mov_Moderately  = ('Steps', lambda x: ((x >= 7500) & (x < 10000)).sum()),
    mov_Light       = ('Steps', lambda x: ((x >= 5000) & (x < 7500 )).sum()),
    mov_Sedentary   = ('Steps', lambda x: ((x >= 0   ) & (x < 5000 )).sum()),


    # Días que tienen registrado su actividad de sueño y si estos son de  sueño saludable ( mínimo 7 horas)
    days_of_sleep  = ('TotalMinutesAsleep', lambda x: (x > 0).sum()),
    healthy_sleep  = ('TotalMinutesAsleep', lambda x: (x > 420).sum()),
 

    # Puntos de actividad
    Combined_Activity_Score    = ('Combined_Activity_Score', 'sum'),
    # Min_Healty_Week_Goal       = ('Combined_Activity_Score',  'count'), #lambda x : x.sum() >= (x.count() * MIN_HEALTH_GOAL_DAILY)),
    # Optimal_Goal_week_Goal     = ('Combined_Activity_Score',  'count'), #lambda x : x.sum() >= (x.count() * OPTIMAL_GOAL_DAILY)),
    # Best_Goal_week_Goal        = ('Combined_Activity_Score',  'count'), #lambda x : x.sum() >= (x.count() * BEST_GOAL_DAILY)),

    # healthy_days = ('Flag_Meets_Best_Goal', lambda x : x.sum() >= 3), # Personas hacen actividad física 
                                                        
    # ############################################# 
    # Días con actividad física y suficiente
    Flag_Meets_Min_Health    = ('Flag_Meets_Min_Health', 'sum'),
    Flag_Meets_Optimal_Goal  = ('Flag_Meets_Optimal_Goal', 'sum'),
    Flag_Meets_Best_Goal     = ('Flag_Meets_Best_Goal', 'sum'),
    Flag_Meets_Less_Min_Goal = ('Flag_Meets_Min_Health', lambda x : (x == 0).sum()),

    # ############################################# 
     
)
test2